# Ultra-Scale Playbook 训练系统 · 第 8/14 课

> 状态：**参考答案版**  
> 一次只完成一课；未通过前不要打开下一课答案。

## 统一完成标准

代码 4 分、Q1～Q3 各 2 分，通过线 8/10。必须解释正确性边界、显存/通信公式中的单位与分片维度；未实际运行的内容只能标记为静态审查。

# 第 8 课：上下文并行 CP

- 对应官方章节：Context Parallelism / Ring Attention / Zig-Zag Ring Attention
- 前置：第 3 课（all-gather / all-to-all 概念）、第 7 课（SP）
- 状态：未开始

## 本课目标

完成后你需要能够：

- 说明 CP 与 SP 的区别：CP 沿序列切分**整个模型**（含 attention），独立于 TP 使用。
- 推演 Ring Attention 的三步循环，说明为什么注意力是唯一需要跨卡通信的模块。
- 解释朴素 Ring Attention 在因果掩码下的负载不均，以及 Zig-Zag 如何均衡。
- 对比 all-gather 与 all-to-all（ring）两种 K/V 交换实现的取舍。

## 核心概念

### 1. CP 解决什么问题

序列极长（128k+）时，即使全量重计算，层边界的激活仍随序列长度线性增长，且 TP 区域内要处理完整序列。CP 把序列沿**整模型**切分到多卡（每卡处理 s/cp 段）。对 MLP/LayerNorm 这类逐 token 模块，切分完全免费；**只有 attention 需要跨卡交换 K/V**——因为每个 token 要 attend 其他所有 token 的 key/value（因果下是全部前序 token）。

### 2. Ring Attention 的三步循环

每步每个 GPU：

1. **异步发送**自己的 K/V 给下一个 GPU（非阻塞）；
2. 用当前持有的 K/V **本地计算**注意力片段（softmax(QKᵀ)·V）；
3. **接收**上一个 GPU 的 K/V，回到第 1 步。

共 cp 步完成整条序列的注意力。通信与计算重叠（理想情况下收包时正好算完）。

### 3. 与 FlashAttention 的关系

两者都依赖 **online softmax**（分块计算 softmax 统计量，不物化完整分数矩阵）：FlashAttention 在单卡内分块 HBM↔SRAM；Ring Attention 在多卡间分块。概念同源（第 13 课实现 online softmax）。

### 4. 因果掩码下的负载不均 → Zig-Zag

朴素切分：rank 0 分到序列开头（token 1–4），rank 3 分到末尾（token 13–16）。

- 因果掩码下，rank 0 的 query 只需要自己这几块 K/V；rank 3 需要**所有**前面 rank 的 K/V。
- 每个 rank 在收到足够的 K/V 后、甚至要等别的 rank 的 block 完成才能做下一次软max汇总 → 负载与等待严重不均。

**Zig-Zag**：chunk 分配不再连续，而是交替从序列两头取——rank 0 拿 {1, 16}，rank 1 拿 {2, 15}……使得每卡都有"早 token + 晚 token"的混合，因果工作量均衡；代价是每个 rank 最终都要从其他所有 rank 拿到 K/V（通信次数略增，但总量相同）。

### 5. 两种 K/V 交换实现

- **All-gather 式**：一次性把所有 K/V 汇总到每卡。实现简单、一步完成；临时显存大（每卡存全部 K/V）。
- **All-to-all（ring）式**：逐块轮转。临时显存小（只多存一块）；多步通信、基延迟略高，但可与计算重叠。
- 教材结论：ring 式更省内存、更优；all-gather 式简单但临时显存大。

### 6. 与其他并行的关系

- CP 后各卡计算的是不同序列片段 → 梯度不同 → 需要像 DP 一样 all-reduce 梯度同步（CP 组内）。
- CP 与 DP/TP/PP/EP 正交，主要针对长序列；DeepSeek 等超长上下文训练用它。
- CP 与 DP 在"输入切分"上相似，区别是 CP 组内要交换 attention 的 K/V。

## 具体演示

4 GPU、4 个 token、各持 1 个 chunk，因果掩码：

- 朴素连续分配：rank 0 只处理 1 对 (q,kv)；rank 3 要处理 4 对 → 负载 1:4。
- Zig-Zag 分配（8 chunk、每卡 2 个）：rank 0 拿 {0,7}、rank 1 拿 {1,6}…每卡工作量相等（9 对）。

## 代码填空题

模拟 chunk 分配与 Ring Attention 轮转，验证 Zig-Zag 的负载均衡。


In [ ]:
def sequential_assignment(num_ranks: int, chunks_per_rank: int):
    """
    朴素连续分配：rank r 拿连续的 chunks_per_rank 个 chunk。
    返回 dict: rank -> chunk 序号列表（chunk 从 0 开始编号）。
    """
    assign = {}
    for r in range(num_ranks):
        assign[r] = ______   # 填空：第 r 个 rank 拿到的 chunk 列表
    return assign


def zigzag_assignment(num_ranks: int, chunks_per_rank: int):
    """
    Zig-Zag：第 r 个 rank 交替从序列两头取。
    序号：rank r 第 k 个 chunk 的全局编号。
    规则：编号 < num_ranks 的 chunk 正序发（chunk r 给 rank r）；
          超过后反向交错：chunk (2*num_ranks - 1 - r) 给 rank r。
    对 chunks_per_rank = 2（最常见的 2 chunk/卡）：rank r 拿 {r, 2p-1-r}。
    """
    total = num_ranks * chunks_per_rank
    assign = {}
    for r in range(num_ranks):
        chunks = []
        for k in range(chunks_per_rank):
            # 填空：交替取数公式（奇数块从尾部取）
            if k % 2 == 0:
                chunks.append(______)      # 正序：第 k/2 轮正序编号
            else:
                chunks.append(______)      # 反序：从尾端回推
        assign[r] = chunks
    return assign


def causal_workload(assign: dict[int, list[int]], num_chunks: int) -> dict[int, int]:
    """
    因果掩码下每个 rank 需要计算的 (q_chunk, kv_chunk) 对数。
    一个 rank 有多个 query chunk 时，工作量 = Σ_q (q 号 + 1)，
    因为 q 可以 attend kv_chunk 0..q（因果）。
    """
    workload = {}
    for r, q_chunks in assign.items():
        total = 0
        for q in q_chunks:
            total += ______                # 填空：该 query chunk 的合法 kv 数量
        workload[r] = total
    return workload


def ring_rotation(num_ranks: int, rank: int, start_chunk: int) -> list[int]:
    """
    Ring Attention 轮转：返回该 rank 在每一步（0..num_ranks-1）持有的
    K/V chunk 编号（K/V 沿环每步前进一个 rank）。
    第 t 步：本 rank 的 K/V 来自上游 t 步前出发的 chunk。
    """
    steps = []
    for t in range(num_ranks):
        steps.append(______)               # 填空：(start + t) 沿环回绕
    return steps


if __name__ == "__main__":
    p = 4
    seq_assign = sequential_assignment(p, 2)
    zz_assign = zigzag_assignment(p, 2)
    print("sequential:", seq_assign)
    print("zig-zag   :", zz_assign)

    w_seq = causal_workload(seq_assign, p * 2)
    w_zz = causal_workload(zz_assign, p * 2)
    print("sequential workload:", w_seq, " max/min =", max(w_seq.values()) / min(w_seq.values()))
    print("zig-zag workload   :", w_zz, " max/min =", max(w_zz.values()) / min(w_zz.values()))
    assert len(set(w_zz.values())) == 1, "zig-zag 应完全均衡"
    print("Zig-Zag 负载完全均衡 ✓")

    # Ring 轮转示例：rank 2 的 K/V 持有序列
    print("rank 2 轮转:", ring_rotation(p, 2, 2))


## 三个问答题


### Q1

CP 中为什么只有 attention 模块需要跨卡通信？为什么交换的是 K/V 而不是 Q？如果把整条序列的 K/V 一次性复制到每卡（all-gather 式），与 ring 式相比显存和延迟各付出什么代价？


### Q2

用因果掩码解释为什么朴素连续切分下"GPU 1 立即算完、GPU 4 要等三轮"，并说明 Zig-Zag 为什么能均衡负载。Zig-Zag 下每个 rank 需要从其他所有 rank 获得 K/V，这是否增加总通信量？


### Q3

教材指出 Ring Attention 与 FlashAttention 都依赖 online softmax。请解释：分块计算注意力时，为什么不能简单地对每个块分别 softmax 再拼接？还需要维护哪些额外统计量？（可以只给概念，第 13 课会用代码实现。）

## 检查与通过标准

总分 10 分：代码正确 4 分（分配公式、工作量统计、轮转公式、均衡验证）、三题各 2 分、通过线 8 分。

一票否决项：

- 分不清 CP 与 SP（本书定义）。
- 认为 CP 中 MLP/LayerNorm 也需要跨卡交换激活。
- 说不出因果掩码导致负载不均的具体机制。
- 认为"分别 softmax 再拼接"是正确做法。


## 参考答案（仅 answer 分支）

先完成题目再核对；核对后必须解释关键公式，并改一个规模重新计算。

In [ ]:
def sequential_assignment(num_ranks: int, chunks_per_rank: int):
    """
    朴素连续分配：rank r 拿连续的 chunks_per_rank 个 chunk。
    返回 dict: rank -> chunk 序号列表（chunk 从 0 开始编号）。
    """
    assign = {}
    for r in range(num_ranks):
        assign[r] = list(range(r * chunks_per_rank, (r + 1) * chunks_per_rank))   # 填空：第 r 个 rank 拿到的 chunk 列表
    return assign


def zigzag_assignment(num_ranks: int, chunks_per_rank: int):
    """
    Zig-Zag：第 r 个 rank 交替从序列两头取。
    序号：rank r 第 k 个 chunk 的全局编号。
    规则：编号 < num_ranks 的 chunk 正序发（chunk r 给 rank r）；
          超过后反向交错：chunk (2*num_ranks - 1 - r) 给 rank r。
    对 chunks_per_rank = 2（最常见的 2 chunk/卡）：rank r 拿 {r, 2p-1-r}。
    """
    total = num_ranks * chunks_per_rank
    assign = {}
    for r in range(num_ranks):
        chunks = []
        for k in range(chunks_per_rank):
            # 填空：交替取数公式（奇数块从尾部取）
            if k % 2 == 0:
                chunks.append((k // 2) * num_ranks + r)      # 正序：第 k/2 轮正序编号
            else:
                chunks.append(total - 1 - (k // 2) * num_ranks - r)      # 反序：从尾端回推
        assign[r] = chunks
    return assign


def causal_workload(assign: dict[int, list[int]], num_chunks: int) -> dict[int, int]:
    """
    因果掩码下每个 rank 需要计算的 (q_chunk, kv_chunk) 对数。
    一个 rank 有多个 query chunk 时，工作量 = Σ_q (q 号 + 1)，
    因为 q 可以 attend kv_chunk 0..q（因果）。
    """
    workload = {}
    for r, q_chunks in assign.items():
        total = 0
        for q in q_chunks:
            total += q + 1                # 填空：该 query chunk 的合法 kv 数量
        workload[r] = total
    return workload


def ring_rotation(num_ranks: int, rank: int, start_chunk: int) -> list[int]:
    """
    Ring Attention 轮转：返回该 rank 在每一步（0..num_ranks-1）持有的
    K/V chunk 编号（K/V 沿环每步前进一个 rank）。
    第 t 步：本 rank 的 K/V 来自上游 t 步前出发的 chunk。
    """
    steps = []
    for t in range(num_ranks):
        steps.append((start_chunk + t) % num_ranks)               # 填空：(start + t) 沿环回绕
    return steps


if __name__ == "__main__":
    p = 4
    seq_assign = sequential_assignment(p, 2)
    zz_assign = zigzag_assignment(p, 2)
    print("sequential:", seq_assign)
    print("zig-zag   :", zz_assign)

    w_seq = causal_workload(seq_assign, p * 2)
    w_zz = causal_workload(zz_assign, p * 2)
    print("sequential workload:", w_seq, " max/min =", max(w_seq.values()) / min(w_seq.values()))
    print("zig-zag workload   :", w_zz, " max/min =", max(w_zz.values()) / min(w_zz.values()))
    assert len(set(w_zz.values())) == 1, "zig-zag 应完全均衡"
    print("Zig-Zag 负载完全均衡 ✓")

    # Ring 轮转示例：rank 2 的 K/V 持有序列
    print("rank 2 轮转:", ring_rotation(p, 2, 2))


# 第 8 课参考答案（复盘用，先提交再打开）

## 补全后的代码（关键填空处）


```python
def sequential_assignment(num_ranks, chunks_per_rank):
    assign[r] = list(range(r * chunks_per_rank, (r + 1) * chunks_per_rank))


def zigzag_assignment(num_ranks, chunks_per_rank):
    total = num_ranks * chunks_per_rank
    for k in range(chunks_per_rank):
        if k % 2 == 0:
            chunks.append((k // 2) * num_ranks + r)          # 正序轮
        else:
            chunks.append(total - 1 - (k // 2) * num_ranks - r)  # 反序轮


def causal_workload(assign, num_chunks):
    total += q + 1        # 因果：query chunk q 可 attend kv chunk 0..q，共 q+1 个


def ring_rotation(num_ranks, rank, start_chunk):
    steps.append((start_chunk + t) % num_ranks)   # 每步 K/V 沿环前进一块
```


运行结果（p=4，每卡 2 chunk）：


```python
sequential: {0:[0,1], 1:[2,3], 2:[4,5], 3:[6,7]}   工作量 {3,7,11,15}，失衡 5 倍
zig-zag   : {0:[0,7], 1:[1,6], 2:[2,5], 3:[3,4]}   工作量 {9,9,9,9}，完全均衡 ✓
rank 2 轮转: [2, 3, 0, 1]
```


## 三个问答题要点


### Q1

- 只有 attention 需要全序列信息（token 要 attend 所有 K/V）；MLP/LN 逐 token 独立，序列切分免费；
- 交换 K/V 而非 Q：每个 token 的 Q 只需自己那份；K/V 是全量共享的"被 attend 对象"；
- all-gather 式：一次通信、简单，但每卡临时显存 = 全部 K/V（O(seq·h) 额外）；ring 式：分块轮转，临时显存只多一块，通信与计算重叠，代价是多步的基延迟。


### Q2

- 因果掩码下 softmax 按行（query）计算：rank 持有序列开头的 query 时，合法 K/V 都在本地或很近；持有末尾 query 的 rank 需要所有前序 K/V，且要等其他 rank 算完才能汇总（online softmax 的 block 间依赖）；
- 朴素分配：rank 0 工作量 1 对、rank 3 工作量 4 对 → 等待链；
- Zig-Zag：每卡都有"早 + 晚"混合的 query → 工作量均衡（{9,9,9,9}）；每个 rank 需要来自其他所有 rank 的 K/V（通信"全连接"），但**总通信量不变**（每块 K/V 仍只传一次/一圈），只是不再能"早退"。


### Q3

- 不能分别 softmax 再拼接：softmax 分母是整行的 exp 和，只看到部分块时不知道全局归一化因子；
- 需要维护：running max m（数值稳定 + 已算块的重新缩放）、running sum l（exp(score−m) 之和）；每来一块：m_new = max(m, rowmax)，旧输出按 exp(m−m_new) 缩放，新块按 exp(score−m_new) 累加；
- 这就是 FlashAttention 的 online softmax（第 13 课代码实现）。

## 通过要点

- CP 沿序列切**整个模型**（可独立使用）；SP 只切 TP 配套的 LN/dropout 区域。
- Ring Attention 三步：async send K/V → 本地计算 → recv；共 cp 步。


## 官方主参考

- [Ultra-Scale Playbook](https://huggingface.co/spaces/nanotron/ultrascale-playbook)
- [PyTorch distributed documentation](https://pytorch.org/docs/stable/distributed.html)